# CDD-11 local intervention-utility pilot

This notebook freezes the A3-M restorer, learns a tiny 32x32-block signed-gain predictor on 15 restoration-training scenes, selects its threshold and the fixed-beta control on 5 different calibration scenes, and reports on the existing 5-scene validation split. It then diagnoses within-image score ranking against matched random rejection without retraining either model. **CDD-11_test is never loaded.**


In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.diagnose_gain_ranking",
    "--checkpoint", str(A3M_CHECKPOINT),
    "--predictor", str(GAIN_ROOT / "gain_predictor.pt"),
    "--calibration", str(GAIN_ROOT / "calibration.json"),
    "--data-root", str(CDD11_ROOT),
    "--output-dir", str(GAIN_ROOT / "ranking_diagnostic"),
    "--random-repeats", "20",
], check=True)


In [ ]:
import json, os, subprocess, zipfile
from pathlib import Path
from IPython.display import FileLink, display

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
A3M_ROOT = Path("/kaggle/working/experiments_a3m")
RUN_NAME = "a3m_multiscale_degradation_sidd32_seed42_20ep"
A3M_CHECKPOINT = A3M_ROOT / RUN_NAME / "best.pt"
GAIN_ROOT = Path("/kaggle/working/gain_predictor")
assert CDD11_ROOT.is_dir() and PRETRAINED_ROOT.is_dir()
gpu_names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), f"Select 2xT4; found {gpu_names}"
print("GPUs:", gpu_names)


In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT), "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/gain_predictor_audit.json",
], check=True)


In [ ]:
# Explicitly approved pilot: train A3-M only when its checkpoint is absent.
if not A3M_CHECKPOINT.is_file():
    subprocess.run([
        "python", "-m", "hybrid_cot_nafnet.run_ablation",
        "--config", "configs/calibration_a3_multiscale.json",
        "--data-root", str(CDD11_ROOT), "--experiments-root", str(A3M_ROOT),
        "--nproc-per-node", "2", "--runs", RUN_NAME,
    ], check=True)
assert A3M_CHECKPOINT.is_file(), f"Missing checkpoint: {A3M_CHECKPOINT}"
print("Frozen restorer:", A3M_CHECKPOINT)


In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.train_gain_predictor",
    "--checkpoint", str(A3M_CHECKPOINT), "--data-root", str(CDD11_ROOT),
    "--output-dir", str(GAIN_ROOT), "--block-size", "32",
    "--blocks-per-image", "64", "--calibration-scenes", "5",
    "--epochs", "12", "--batch-size", "256",
], check=True)


In [ ]:
import pandas as pd
summary = json.loads((GAIN_ROOT / "summary.json").read_text())
display(summary)
display(pd.read_csv(GAIN_ROOT / "validation_metrics.csv"))
ranking_summary = json.loads((GAIN_ROOT / "ranking_diagnostic" / "summary.json").read_text())
display(ranking_summary)
display(pd.DataFrame(ranking_summary["curves"]))
archive = Path("/kaggle/working/gain_predictor_results.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
    for path in sorted(GAIN_ROOT.rglob("*")):
        if path.is_file():
            output_zip.write(path, path.relative_to(GAIN_ROOT.parent))
display(FileLink(str(archive)))
